# Taylor blast wave from a photograph sequence

This notebook explains the **first question** on the Balliol / UNIQ physics sheet using only:
- dimensional analysis
- algebra
- scaling ideas
- careful use of a photograph sequence

It **does not** use partial differential equations or the Navier–Stokes equations.

The original question asks you to model the radius of a spherical shock front after a sudden release of energy in air. fileciteturn0file0

## Why this is a famous problem

G. I. Taylor showed that if an explosion releases energy $E$ into air of density $\rho$, then the shock radius $r$ after time $t$ must have the form

$$r \propto \left(\frac{E t^2}{\rho}\right)^{1/5}.$$

That is remarkable because the shape of the answer can be obtained **without solving the full fluid dynamics problem**.

Even better, if you have a sequence of photographs with a time stamp, a distance scale, and a visible fireball radius, you can work **backwards** and estimate the explosion energy.

## What we will do in this notebook

1. Re-derive the scaling law for the blast radius.
2. Derive the scaling of front speed and pressure.
3. Use the uploaded photograph sequence to estimate a few radii.
4. Use those measurements to get an order-of-magnitude estimate for the explosion energy.
5. Reflect on what this problem illustrates about dimensional analysis.

## 1. Dimensional analysis for the radius

Assume the shock radius depends only on the explosion energy $E$, the air density $\rho$, and the time $t$. Write

$$r \propto E^a \rho^b t^c.$$

The dimensions are $[r] = L$, $[E] = ML^2T^{-2}$, $[\rho] = ML^{-3}$, and $[t] = T$.

In [ ]:
import sympy as sp

a, b, c = sp.symbols('a b c', real=True)

# Match powers of M, L, T
eq_M = sp.Eq(a + b, 0)
eq_L = sp.Eq(2*a - 3*b, 1)
eq_T = sp.Eq(-2*a + c, 0)

solution = sp.solve([eq_M, eq_L, eq_T], [a, b, c], dict=True)[0]
solution

So $a = \tfrac{1}{5}$, $b = -\tfrac{1}{5}$, $c = \tfrac{2}{5}$, giving

$$\boxed{r = C\left(\frac{E t^2}{\rho}\right)^{1/5}},$$

where $C$ is a dimensionless constant that dimensional analysis alone cannot determine.

## 2. Speed and pressure scales

### Front speed

Since $r \propto t^{2/5}$, differentiating gives

$$u = \frac{dr}{dt} \propto t^{-3/5}.$$

Restoring the dependence on $E$ and $\rho$,

$$\boxed{u \propto E^{1/5}\rho^{-1/5}t^{-3/5}}.$$

### Pressure scale

Pressure has dimensions $[p] = ML^{-1}T^{-2}$. Assume $p \propto E^a\rho^b t^c$ and solve by dimensional matching.

In [ ]:
a, b, c = sp.symbols('a b c', real=True)

eq_M = sp.Eq(a + b, 1)
eq_L = sp.Eq(2*a - 3*b, -1)
eq_T = sp.Eq(-2*a + c, -2)

pressure_solution = sp.solve([eq_M, eq_L, eq_T], [a, b, c], dict=True)[0]
pressure_solution

This gives $a = \tfrac{2}{5}$, $b = \tfrac{3}{5}$, $c = -\tfrac{6}{5}$, so

$$\boxed{p \propto E^{2/5}\rho^{3/5}t^{-6/5}}.$$

The pressure falls quickly with time, which matches the idea that the blast is strongest early on.

## 3. Use the photograph sequence

The uploaded image is exactly the kind of evidence that makes this problem so memorable:
- the frames are time-stamped in milliseconds
- there is a scale bar labelled **100 metres**
- the fireball edge is visible enough for an approximate radius measurement

The aim is not perfect image processing. The aim is to show how a physicist can turn a photograph into a quantitative estimate.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

img = Image.open('blast.png').convert('RGB')
img_arr = np.array(img)

plt.figure(figsize=(12, 10))
plt.imshow(img_arr)
plt.axis('off')
plt.title('Explosion contact sheet — time stamps and 100 m scale bar visible')
plt.tight_layout()
plt.show()

print(f"Image size: {img.size[0]} × {img.size[1]} px")

### Approximate calibration from the scale bar

From the uploaded image, the 100 m scale bar is about **42 pixels** long, giving

$$1\text{ pixel} \approx \frac{100}{42}\text{ m} \approx 2.38\text{ m}.$$

That is already good enough for an order-of-magnitude estimate.

In [ ]:
# Approximate calibration from a manual measurement of the scale bar
scale_bar_pixels = 42
metres_per_pixel = 100 / scale_bar_pixels
metres_per_pixel

### Approximate fireball radii from a few frames

Below is a **rough manual measurement set** from several frames.
The radii are measured from the centre of the dome to the edge in pixels, then converted to metres.

These are deliberately approximate. The educational point is that the scaling law is robust enough that even rough data are useful.

In [ ]:
import pandas as pd

# Approximate manual measurements from the uploaded photo sequence
# times in milliseconds; radius_px is a single characteristic radius from each frame
data = pd.DataFrame({
    'time_ms': [15.1, 17.0, 18.8, 20.7, 22.6, 24.4],
    'radius_px': [58, 60, 62, 64, 66, 68]
})

data['time_s'] = data['time_ms'] / 1000
data['radius_m'] = data['radius_px'] * metres_per_pixel

# Best-fit curve with the theoretical scaling retained: r = k t^(2/5).
# Only the constant k is fitted from the data.
k_fit = np.sum(data['radius_px'] * data['time_ms']**0.4) / np.sum(data['time_ms']**0.8)
tolerance_px = 1.5  # estimated manual measurement uncertainty

data['radius_px_pred'] = k_fit * data['time_ms']**0.4
data['residual_px'] = data['radius_px'] - data['radius_px_pred']
data['abs_error_px'] = np.abs(data['residual_px'])
data['radius_m_pred'] = data['radius_px_pred'] * metres_per_pixel
data['abs_error_m'] = data['radius_m_pred'] - data['radius_m']
data['pct_error'] = 100 * data['abs_error_m'] / data['radius_m']
data

These values are not meant to be exact. They are just plausible measurements from the image, enough to demonstrate the method.
Instead of forcing a curve onto the photo, we keep the theoretical scaling $r \propto t^{2/5}$ and fit only the constant $k$ to all the measured frames.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data['time_s'], data['radius_m'], label='measured radius')
plt.plot(data['time_s'], data['radius_m_pred'], label='best-fit theory with fixed $t^{2/5}$ scaling')
plt.xlabel('time / s')
plt.ylabel('radius / m')
plt.title('Measured radius compared with best-fit theory retaining $t^{2/5}$ scaling')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data['time_ms'], data['radius_px'], label='measured radius_px')
plt.plot(data['time_ms'], data['radius_px_pred'], label='best-fit predicted radius_px')
plt.fill_between(
    data['time_ms'],
    data['radius_px_pred'] - tolerance_px,
    data['radius_px_pred'] + tolerance_px,
    alpha=0.2,
    label=f'prediction ± {tolerance_px:.1f} px tolerance'
)
plt.xlabel('time / ms')
plt.ylabel('radius / px')
plt.title('Measured and best-fit predicted characteristic radius in pixels')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
metrics = pd.Series({
    'tolerance_px': tolerance_px,
    'mean_abs_error_px': data['abs_error_px'].mean(),
    'rmse_px': np.sqrt(np.mean(data['residual_px']**2)),
    'mean_abs_percent_error': np.mean(np.abs(data['pct_error'])),
    'max_abs_percent_error': np.max(np.abs(data['pct_error'])),
    'points_within_tolerance': int((data['abs_error_px'] <= tolerance_px).sum()),
    'total_points': int(len(data)),
    'fraction_within_tolerance': (data['abs_error_px'] <= tolerance_px).mean(),
})
metrics

In [ ]:
plt.figure(figsize=(7, 4))
plt.axhline(0, color='black', linewidth=1)
plt.axhline(tolerance_px, color='tab:red', linestyle='--', label=f'+{tolerance_px:.1f} px tolerance')
plt.axhline(-tolerance_px, color='tab:red', linestyle='--', label=f'-{tolerance_px:.1f} px tolerance')
plt.scatter(data['time_ms'], data['residual_px'], color='tab:purple', label='pixel residual')
plt.plot(data['time_ms'], data['residual_px'], color='tab:purple', alpha=0.6)
plt.xlabel('time / ms')
plt.ylabel('residual / px')
plt.title('Residuals: measured radius_px minus best-fit prediction')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
comparison = data[[
    'time_ms', 'radius_px', 'radius_px_pred', 'residual_px', 'abs_error_px', 'radius_m', 'radius_m_pred', 'abs_error_m', 'pct_error'
]].copy()
comparison.columns = [
    'time_ms', 'measured_radius_px', 'predicted_radius_px', 'residual_px', 'abs_error_px', 'measured_radius_m', 'predicted_radius_m', 'error_m', 'error_percent'
]
comparison

## 4. Estimate the explosion energy

From $r = C\!\left(\dfrac{E t^2}{\rho}\right)^{1/5}$ we get $E = \dfrac{\rho r^5}{C^5 t^2}$. Setting $C \sim 1$ gives the order-of-magnitude estimate

$$E \sim \frac{\rho r^5}{t^2}.$$

That is exactly what dimensional analysis is for.

In [ ]:
rho = 1.2  # kg m^-3, approximate density of air

data['E_est_J'] = rho * data['radius_m']**5 / data['time_s']**2
data['E_est_kilotons_TNT'] = data['E_est_J'] / 4.184e12

summary = data[['time_ms', 'radius_m', 'E_est_J', 'E_est_kilotons_TNT']].copy()
summary

In [ ]:
E_mean = data['E_est_J'].mean()
kt_mean = E_mean / 4.184e12
E_mean, kt_mean

The estimates will not be identical because:
- the image measurements are rough
- the edge is not perfectly sharp
- we ignored the constant $C$
- a fireball is not exactly the same thing as the shock front

But the answer still lands in the right **large-scale energy range**, which is the striking part.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data['time_s']**2, data['radius_m']**5)
plt.xlabel(r'$t^2$ / s$^2$')
plt.ylabel(r'$r^5$ / m$^5$')
plt.title(r'Checking the scaling law: if $r^5 \propto t^2$, the graph should be roughly linear')
plt.grid(True, alpha=0.3)
plt.show()

Why plot $r^5$ against $t^2$? Because the scaling law $r \propto \left(\tfrac{E t^2}{\rho}\right)^{1/5}$ implies $r^5 \propto t^2$, so a roughly straight-line trend supports the model.

## 5. Does making the bomb bigger matter much?

This is a lovely scaling question.

The blast radius scales as $r \propto E^{1/5}$, so if the energy doubles the radius increases by only $2^{1/5} \approx 1.15$ — about **15%**.

Making the bomb much more energetic does **not** make the visible radius increase by the same factor. That fifth root is very weak growth.

In [ ]:
for factor in [2, 4, 8, 10, 100]:
    print(f"Energy x {factor:>3}  ->  radius x {factor**(1/5):.3f}")

## 6. What to take from this problem

This problem is hard in **ideas**, not in advanced maths.

The key insights are:

- The problem is asking for a **model**, not an exact fluid solution.
- The only relevant quantities given are **energy**, **density**, and **time**.
- Dimensions can force the form of the answer.
- Real photographs can be turned into measurements.
- Physics often gives useful answers before exact equations are solved.

That is why this is such a famous example of dimensional analysis.

## 7. Short exam-style summary

Assume the shock radius depends only on energy $E$, density $\rho$, and time $t$:

$$r \propto E^a \rho^b t^c.$$

Matching dimensions gives $a = \tfrac{1}{5}$, $b = -\tfrac{1}{5}$, $c = \tfrac{2}{5}$, so

$$r \propto \left(\frac{E t^2}{\rho}\right)^{1/5}.$$

Differentiating: $u = dr/dt \propto E^{1/5}\rho^{-1/5}t^{-3/5}$. Pressure scales as $p \propto E^{2/5}\rho^{3/5}t^{-6/5}$.

Since $r \propto E^{1/5}$, increasing the bomb energy has only a weak effect on radius. Given a time-stamped film, the energy can be estimated as $E \sim \rho r^5/t^2$ up to a dimensionless constant.